## Бонусный эксперимент: идея совмещения обучения без учителя (кластеризацию) с анализом мошенничества (выясление аномальных паттернов)

Теория: Фрод - это нетипичное сочетание признаков. Обычные транзакции легко разделимы на тепичные паттерны, а не типичные чаще будут фродом, чем нет. 
Подход: С помощью кластеризации создаём кластеры "архетипов" поведения клиентов, мерчантов, транзакций, ошибок и профилей. Затем обучаем модель логистической регрессии исключительно на данных о кластеризации: метках кластеров и расстояния для центроидов. Если теория верна - модель покажет результат превышающий показатели baseline. Мошенничество будет выявлено т.к. оно будет сосредоточено в специфических кластеров либо будет результатом не типичного поведения - будет иметься большое расстояние от центроида кластера то соответвующего признака. 

In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

In [2]:
# Загружаем данные
data = pd.read_parquet('data_after_FE.parquet')
with open('data_after_FE_schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)

# сначала datetime
for col in schema['datetime_cols']:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

# потом category
for col in schema['category_cols']:
    if col in data.columns:
        data[col] = data[col].astype('category')

# потом остальные типы
for col, dtype_str in schema['dtypes'].items():
    if col not in data.columns:
        continue
    
    if col in schema['datetime_cols'] or col in schema['category_cols']:
        continue
    
    try:
        if dtype_str == 'object':
            data[col] = data[col].astype('string')
        else:
            data[col] = data[col].astype(dtype_str)
    except Exception as e:
        print(f'Не удалось привести {col} к {dtype_str}: {e}')
mask = data['Timestamp'].between('2015-01-01', '2019-10-31 23:59:59')
data = data.loc[mask].copy()
print(data.info())

<class 'pandas.core.frame.DataFrame'>
Index: 7890638 entries, 14656146 to 22546783
Data columns (total 63 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   User                                 int16         
 1   Card                                 int8          
 2   Timestamp                            datetime64[ns]
 3   Amount                               float32       
 4   Use_Chip                             int8          
 5   Is_Online                            int8          
 6   Merchant_ID                          int32         
 7   Merchant_State                       category      
 8   MCC                                  int16         
 9   Has_Error                            int8          
 10  Fraud                                int8          
 11  Gender                               int8          
 12  Is_Apartment                         int8          
 13  Total_Debt              

In [3]:
data.shape

(7890638, 63)

In [ ]:


# =========================================================
# 0. БАЗОВЫЕ НАСТРОЙКИ
# =========================================================

TARGET_COL = "Fraud"
TIME_COL = "Timestamp"
RANDOM_STATE = 42

# Для отладки, в рабочем режиме True
USE_RATE_FEATURES = True

# На память:
# OneHot не используем
# Для категорий OrdinalEncoder.
# Для кластеризации на больших данных  MiniBatchKMeans вместо обычного KMeans.

# =========================================================
# 1. СОРТИРОВКА И SPLIT ПО ВРЕМЕНИ
# =========================================================

data = data.sort_values(TIME_COL).reset_index(drop=True)

train_end = int(len(data) * 0.70)
valid_end = int(len(data) * 0.85)

train_df = data.iloc[:train_end].copy()
valid_df = data.iloc[train_end:valid_end].copy()
test_df  = data.iloc[valid_end:].copy()

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL].astype("int8")

X_valid = valid_df.drop(columns=[TARGET_COL])
y_valid = valid_df[TARGET_COL].astype("int8")

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL].astype("int8")

print("TRAIN:", X_train.shape, y_train.shape)
print("VALID:", X_valid.shape, y_valid.shape)
print("TEST :", X_test.shape, y_test.shape)

# =========================================================
# 2. НАБОРЫ ПРИЗНАКОВ ДЛЯ РАЗНЫХ КЛАСТЕРНЫХ ПРОСТРАНСТВ
# =========================================================
# Не включаем:
# - User
# - Card
# - Merchant_ID
# - Timestamp
# - признаки, являющиеся заменой временному штампу (возвраст аккаунта и подобные)
# Потому что это либо ID, либо сырой time-stamp.

cluster_feature_sets = {
    "transaction_profile": {
        "num": [
            "Amount",
            "amount_log",
            "txn_hour",
            "txn_dayofweek",
            "txn_day",
            "hour_sin",
            "hour_cos",
            "dow_sin",
            "dow_cos",
        ],
        "cat": [
            "Use_Chip",
            "Is_Online",
            "Has_Error",
            "Merchant_State",
            "MCC",
            "Card_Brand",
            "Card_Type",
            "is_weekend",
            "is_night",
            "is_business_hours",
            "amount_round_10",
            "txn_gap_bin",
            "prev_not_foreign_card",
            "is_foreign_offline",
        ]
    },

    "customer_profile": {
        "num": [
            "Total_Debt",
            "FICO",
            "Credit_Limit",
            "Amount_to_Income",
            "user_age",
        ],
        "cat": [
            "Gender",
            "Is_Apartment",
            "Num_Credit_Cards",
            "Has_Chip",
            "Cards_Issued",
        ]
    },

    "behavior_profile": {
        "num": [
            "time_since_prev_txn_card_min",
            "time_since_prev_txn_user_min",
            "errors_prev_1h",
            "txn_count_5m_card",
            "txn_count_1h_card",
            "txn_count_5m_user",
            "txn_count_1h_user",
            "merchant_velocity_ratio_log",
        ],
        "cat": [
            "first_user_payment_to_this_merchant",
            "state_changed_1d",
            "card_burst_5m",
            "txn_gap_bin",
            "prev_not_foreign_card",
            "is_foreign_offline",
            "is_night",
            "is_weekend",
        ]
    },

    "merchant_profile": {
        "num": [
            "merchant_txn_count_24h",
            "merchant_amount_sum_24h",
            "merchant_txn_count_1h",
            "merchant_amount_sum_1h",
            "merchant_velocity_ratio_log",
        ],
        "cat": [
            "Merchant_State",
            "MCC",
            "Is_Online",
            "Use_Chip",
            "Has_Error",
            "first_user_payment_to_this_merchant",
        ]
    },

    "error_profile": {
        "num": [
            "errors_prev_1h",
        ],
        "cat": [
            "Has_Error",
            "Error_Bad_CVV",
            "Error_Bad_Card_Number",
            "Error_Bad_Expiration",
            "Error_Bad_PIN",
            "Error_Bad_Zipcode",
            "Error_Insufficient_Balance",
            "Error_Technical_Glitch",
        ]
    }
}

if USE_RATE_FEATURES:
    cluster_feature_sets["risk_context_profile"] = {
        "num": [
            "merchant_fraud_rate",
            "state_fraud_rate",
        ],
        "cat": [
            "Merchant_State",
            "MCC",
            "Is_Online",
            "Has_Error",
        ]
    }

# Оставляем только реально существующие колонки
for set_name, cfg in cluster_feature_sets.items():
    cfg["num"] = [c for c in cfg["num"] if c in X_train.columns]
    cfg["cat"] = [c for c in cfg["cat"] if c in X_train.columns]

print("\nИспользуемые наборы признаков:")
for set_name, cfg in cluster_feature_sets.items():
    print(f"{set_name}: num={len(cfg['num'])}, cat={len(cfg['cat'])}")

# =========================================================
# 3. ПРЕПРОЦЕССОР ДЛЯ КЛАСТЕРИЗАЦИИ
# =========================================================

def make_cluster_preprocessor(num_cols, cat_cols):
    transformers = []

    if num_cols:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("num", num_pipe, num_cols))

    if cat_cols:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ])
        transformers.append(("cat", cat_pipe, cat_cols))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

# =========================================================
# 4. Обучение КЛАСТЕРИЗАТОРА
# =========================================================

def fit_cluster_model(
    X_train_part,
    num_cols,
    cat_cols,
    n_clusters=20,
    batch_size=10000,
    fit_sample_size=300_000,
    random_state=42
):
    preprocessor = make_cluster_preprocessor(num_cols, cat_cols)

    # Для ускорения fit кластеризатора берём подвыборку train
    if fit_sample_size is not None and len(X_train_part) > fit_sample_size:
        fit_idx = X_train_part.sample(
            n=fit_sample_size,
            random_state=random_state,
            replace=False
        ).index
        X_fit = X_train_part.loc[fit_idx]
    else:
        X_fit = X_train_part

    X_fit_prepared = preprocessor.fit_transform(X_fit)

    clusterer = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=batch_size,
        n_init="auto"
    )
    clusterer.fit(X_fit_prepared)

    return preprocessor, clusterer

# =========================================================
# 5. ПРЕВРАЩЕНИЕ ДАННЫХ В КЛАСТЕРНЫЕ ПРИЗНАКИ
# =========================================================

def transform_cluster_features(preprocessor, clusterer, X_part, prefix, add_distances=True):
    X_prepared = preprocessor.transform(X_part)

    out = pd.DataFrame(index=X_part.index)

    cluster_ids = clusterer.predict(X_prepared)
    out[f"{prefix}_cluster"] = cluster_ids.astype("int16")

    if add_distances:
        distances = clusterer.transform(X_prepared)
        dist_cols = [f"{prefix}_dist_{i}" for i in range(distances.shape[1])]
        dist_df = pd.DataFrame(distances, index=X_part.index, columns=dist_cols)
        out = pd.concat([out, dist_df], axis=1)

    return out

# =========================================================
# 6. ОБУЧАЕМ НЕСКОЛЬКО КЛАСТЕРИЗАТОРОВ
# =========================================================

cluster_models = {}
cluster_feature_frames_train = []
cluster_feature_frames_valid = []
cluster_feature_frames_test = []

n_clusters_map = {
    "transaction_profile": 35,
    "customer_profile": 20,
    "behavior_profile": 30,
    "merchant_profile": 30,
    "error_profile": 10,
    "risk_context_profile": 20
}

for set_name, cfg in cluster_feature_sets.items():
    num_cols = cfg["num"]
    cat_cols = cfg["cat"]

    if len(num_cols) == 0 and len(cat_cols) == 0:
        print(f"Пропуск {set_name}: пустой набор признаков")
        continue

    used_cols = num_cols + cat_cols

    print(f"\nFitting clusterer: {set_name}")
    print("Columns:", used_cols)

    preprocessor, clusterer = fit_cluster_model(
        X_train_part=X_train[used_cols],
        num_cols=num_cols,
        cat_cols=cat_cols,
        n_clusters=n_clusters_map.get(set_name, 20),
        batch_size=10000,
        fit_sample_size=300_000,
        random_state=RANDOM_STATE
    )

    cluster_models[set_name] = {
        "preprocessor": preprocessor,
        "clusterer": clusterer,
        "num_cols": num_cols,
        "cat_cols": cat_cols
    }

    train_cluster_df = transform_cluster_features(
        preprocessor, clusterer, X_train[used_cols], set_name, add_distances=True
    )
    valid_cluster_df = transform_cluster_features(
        preprocessor, clusterer, X_valid[used_cols], set_name, add_distances=True
    )
    test_cluster_df = transform_cluster_features(
        preprocessor, clusterer, X_test[used_cols], set_name, add_distances=True
    )

    cluster_feature_frames_train.append(train_cluster_df)
    cluster_feature_frames_valid.append(valid_cluster_df)
    cluster_feature_frames_test.append(test_cluster_df)

# =========================================================
# 7. СОБИРАЕМ DF ТОЛЬКО ИЗ КЛАСТЕРНЫХ ПРИЗНАКОВ
# =========================================================

X_train_clusters = pd.concat(cluster_feature_frames_train, axis=1)
X_valid_clusters = pd.concat(cluster_feature_frames_valid, axis=1)
X_test_clusters  = pd.concat(cluster_feature_frames_test, axis=1)

print("\nCluster feature space:")
print("TRAIN:", X_train_clusters.shape)
print("VALID:", X_valid_clusters.shape)
print("TEST :", X_test_clusters.shape)

display(X_train_clusters.head())

# =========================================================
# 8. ОБУЧАЕМ МОДЕЛЬ НА КЛАСТЕРНЫХ ПРИЗНАКАХ
# =========================================================

cluster_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

cluster_model.fit(X_train_clusters, y_train)

valid_proba = cluster_model.predict_proba(X_valid_clusters)[:, 1]
test_proba = cluster_model.predict_proba(X_test_clusters)[:, 1]

# =========================================================
# 9. МЕТРИКИ
# =========================================================

def evaluate_threshold_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0)
    }

def find_best_f1_threshold(y_true, proba, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)

    rows = []
    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        rows.append({
            "threshold": thr,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)
        })

    df_thr = pd.DataFrame(rows)
    best_row = df_thr.loc[df_thr["f1"].idxmax()]
    return df_thr, best_row

print("\n=== VALIDATION ===")
print("PR-AUC :", average_precision_score(y_valid, valid_proba))
print("ROC-AUC:", roc_auc_score(y_valid, valid_proba))

thr_table_valid, best_valid_thr = find_best_f1_threshold(y_valid, valid_proba)

print("\nЛучший порог на VALID по F1:")
print(best_valid_thr)

best_thr = float(best_valid_thr["threshold"])

valid_metrics = evaluate_threshold_metrics(y_valid, valid_proba, threshold=best_thr)
test_metrics  = evaluate_threshold_metrics(y_test, test_proba, threshold=best_thr)

print("\n=== VALID @ BEST THRESHOLD ===")
for k, v in valid_metrics.items():
    print(f"{k}: {v}")

print("\n=== TEST ===")
print("PR-AUC :", average_precision_score(y_test, test_proba))
print("ROC-AUC:", roc_auc_score(y_test, test_proba))
for k, v in test_metrics.items():
    print(f"{k}: {v}")

TRAIN: (5523446, 62) (5523446,)
VALID: (1183596, 62) (1183596,)
TEST : (1183596, 62) (1183596,)

Используемые наборы признаков:
transaction_profile: num=9, cat=14
customer_profile: num=5, cat=5
behavior_profile: num=8, cat=8
merchant_profile: num=5, cat=6
error_profile: num=1, cat=8
risk_context_profile: num=2, cat=4

Fitting clusterer: transaction_profile
Columns: ['Amount', 'amount_log', 'txn_hour', 'txn_dayofweek', 'txn_day', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'Use_Chip', 'Is_Online', 'Has_Error', 'Merchant_State', 'MCC', 'Card_Brand', 'Card_Type', 'is_weekend', 'is_night', 'is_business_hours', 'amount_round_10', 'txn_gap_bin', 'prev_not_foreign_card', 'is_foreign_offline']

Fitting clusterer: customer_profile
Columns: ['Total_Debt', 'FICO', 'Credit_Limit', 'Amount_to_Income', 'user_age', 'Gender', 'Is_Apartment', 'Num_Credit_Cards', 'Has_Chip', 'Cards_Issued']

Fitting clusterer: behavior_profile
Columns: ['time_since_prev_txn_card_min', 'time_since_prev_txn_user_min', '

,transaction_profile_cluster,transaction_profile_dist_0,transaction_profile_dist_1,transaction_profile_dist_2,transaction_profile_dist_3,transaction_profile_dist_4,transaction_profile_dist_5,transaction_profile_dist_6,transaction_profile_dist_7,transaction_profile_dist_8,...,risk_context_profile_dist_10,risk_context_profile_dist_11,risk_context_profile_dist_12,risk_context_profile_dist_13,risk_context_profile_dist_14,risk_context_profile_dist_15,risk_context_profile_dist_16,risk_context_profile_dist_17,risk_context_profile_dist_18,risk_context_profile_dist_19
0,2,54.248403,104.552756,4.410942,105.938425,43.163537,37.039382,84.529333,74.106834,45.028270,...,16.383395,118.242208,33.949058,110.645563,105.603754,39.694362,78.619819,51.443720,85.850227,36.797458
1,32,35.061739,18.848103,86.462452,41.797384,51.030995,50.648035,15.674890,13.485410,52.881915,...,94.037641,32.090471,91.756117,28.492631,25.419471,59.766870,16.112952,55.755108,23.379909,50.428469
2,25,15.573787,49.571578,55.985433,61.502515,22.181967,22.558793,33.807205,18.887513,32.155523,...,65.095162,63.101664,62.029241,55.158185,53.439638,37.222814,28.221363,31.252145,32.900675,22.063523
3,3,53.097511,36.372779,105.354627,4.656307,82.006106,69.662305,28.342788,48.189140,61.366110,...,106.924017,40.352538,120.884533,56.514897,21.274483,67.772139,32.311888,90.717493,63.498067,69.510661
4,9,49.156108,16.697055,103.197839,21.683525,72.767518,66.391840,19.149552,35.342416,62.837130,...,107.847498,22.717446,113.269989,36.535508,2.964844,69.864600,24.431405,79.112396,45.918015,66.221744


c:\Users\dmytr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



=== VALIDATION ===
PR-AUC : 0.34127406313973385
ROC-AUC: 0.9710444554728014

Лучший порог на VALID по F1:
threshold    0.990000
precision    0.273483
recall       0.565616
f1           0.368696
Name: 98, dtype: float64

=== VALID @ BEST THRESHOLD ===
accuracy: 0.9971442958577083
precision: 0.2734829592684954
recall: 0.5656160458452723
f1: 0.3686963018304072

=== TEST ===
PR-AUC : 0.471007485087666
ROC-AUC: 0.9951024081050692
accuracy: 0.9944178587964136
precision: 0.19444444444444445
recall: 0.8569032979318055
f1: 0.31696474723457047


In [5]:
results = []

results.append({
    "Model": "Cluster LogReg",
    "VALID_PR_AUC": average_precision_score(y_valid, valid_proba),
    "VALID_ROC_AUC": roc_auc_score(y_valid, valid_proba),
    "TEST_PR_AUC": average_precision_score(y_test, test_proba),
    "TEST_ROC_AUC": roc_auc_score(y_test, test_proba),
    "Best_Threshold": best_thr,
    "TEST_Precision": test_metrics["precision"],
    "TEST_Recall": test_metrics["recall"],
    "TEST_F1": test_metrics["f1"]
})

pd.DataFrame(results)

,Model,VALID_PR_AUC,VALID_ROC_AUC,TEST_PR_AUC,TEST_ROC_AUC,Best_Threshold,TEST_Precision,TEST_Recall,TEST_F1
0,Cluster LogReg,0.341274,0.971044,0.471007,0.995102,0.99,0.194444,0.856903,0.316965


In [6]:
cluster_model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42
)

cluster_model.fit(X_train_clusters, y_train)

valid_proba = cluster_model.predict_proba(X_valid_clusters)[:, 1]
test_proba  = cluster_model.predict_proba(X_test_clusters)[:, 1]

print("VALID PR-AUC :", average_precision_score(y_valid, valid_proba))
print("VALID ROC-AUC:", roc_auc_score(y_valid, valid_proba))
print("TEST  PR-AUC :", average_precision_score(y_test, test_proba))
print("TEST  ROC-AUC:", roc_auc_score(y_test, test_proba))

c:\Users\dmytr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


VALID PR-AUC : 0.35294918366516537
VALID ROC-AUC: 0.9730119291602507
TEST  PR-AUC : 0.47869354965141625
TEST  ROC-AUC: 0.9943378098229367


In [7]:
def find_best_f1_threshold(y_true, proba):
    thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
    rows = []

    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        rows.append({
            "threshold": thr,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
        })

    df_thr = pd.DataFrame(rows)
    best_row = df_thr.loc[df_thr["f1"].idxmax()]
    return df_thr, best_row

thr_table_valid, best_valid_thr = find_best_f1_threshold(y_valid, valid_proba)
print(best_valid_thr)

best_thr = float(best_valid_thr["threshold"])

valid_pred = (valid_proba >= best_thr).astype(int)
test_pred  = (test_proba >= best_thr).astype(int)

print("\nVALID")
print("Accuracy :", accuracy_score(y_valid, valid_pred))
print("Precision:", precision_score(y_valid, valid_pred, zero_division=0))
print("Recall   :", recall_score(y_valid, valid_pred, zero_division=0))
print("F1       :", f1_score(y_valid, valid_pred, zero_division=0))

print("\nTEST")
print("Accuracy :", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred, zero_division=0))
print("Recall   :", recall_score(y_test, test_pred, zero_division=0))
print("F1       :", f1_score(y_test, test_pred, zero_division=0))

threshold    0.990000
precision    0.270542
recall       0.554728
f1           0.363705
Name: 98, dtype: float64

VALID
Accuracy : 0.9971383816775319
Precision: 0.2705422023476803
Recall   : 0.5547277936962751
F1       : 0.3637046778132632

TEST
Accuracy : 0.9949247885258146
Precision: 0.20870165745856353
Recall   : 0.8446059250978201
F1       : 0.334699302248311


In [ ]:
# Попробуем градинетный бустинг - древесная модель может быть интереснее, т.к. у нас комбинация признаков - принадлежность к кластеру
# и центроиды кластеров - может быть нелинейная зависимость между этими признаками и таргетом, которую логрегрессия не сможет уловить.
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

xgb_cluster_model = XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    random_state=42,
    n_jobs=1,
    device="cuda",
    tree_method="hist",
    early_stopping_rounds=200,
)

xgb_cluster_model.fit(
    X_train_clusters,
    y_train,
    eval_set=[(X_valid_clusters, y_valid)],
    verbose=50,
)

valid_proba_xgb = xgb_cluster_model.predict_proba(X_valid_clusters)[:, 1]
test_proba_xgb  = xgb_cluster_model.predict_proba(X_test_clusters)[:, 1]

print("VALID PR-AUC :", average_precision_score(y_valid, valid_proba_xgb))
print("VALID ROC-AUC:", roc_auc_score(y_valid, valid_proba_xgb))
print("TEST  PR-AUC :", average_precision_score(y_test, test_proba_xgb))
print("TEST  ROC-AUC:", roc_auc_score(y_test, test_proba_xgb))

[0]	validation_0-aucpr:0.00757
[50]	validation_0-aucpr:0.04600
[100]	validation_0-aucpr:0.13708
[150]	validation_0-aucpr:0.21087
[200]	validation_0-aucpr:0.25143
[250]	validation_0-aucpr:0.27790
[300]	validation_0-aucpr:0.30981
[350]	validation_0-aucpr:0.32416
[400]	validation_0-aucpr:0.34687
[450]	validation_0-aucpr:0.36026
[500]	validation_0-aucpr:0.37690
[550]	validation_0-aucpr:0.39214
[600]	validation_0-aucpr:0.40587
[650]	validation_0-aucpr:0.41695
[700]	validation_0-aucpr:0.42627
[750]	validation_0-aucpr:0.43266
[800]	validation_0-aucpr:0.44227
[850]	validation_0-aucpr:0.44999
[900]	validation_0-aucpr:0.45346
[950]	validation_0-aucpr:0.45944
[999]	validation_0-aucpr:0.46387


c:\Users\dmytr\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:774: UserWarning: [20:22:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


VALID PR-AUC : 0.4641087632463155
VALID ROC-AUC: 0.992541450671186
TEST  PR-AUC : 0.4157466359899382
TEST  ROC-AUC: 0.993946463040691


In [19]:
# Найти лучший порог по F1 для XGBoost
thr_table_xgb, best_valid_thr_xgb = find_best_f1_threshold(y_valid, valid_proba_xgb)
print(best_valid_thr_xgb)
best_thr_xgb = float(best_valid_thr_xgb["threshold"])

# Метрики на тесте при этом пороге
test_pred_xgb = (test_proba_xgb >= best_thr_xgb).astype(int)
test_metrics_xgb = {
    "precision": precision_score(y_test, test_pred_xgb, zero_division=0),
    "recall": recall_score(y_test, test_pred_xgb, zero_division=0),
    "f1": f1_score(y_test, test_pred_xgb, zero_division=0)
}
results.append({
    "Model": "Cluster XGB",
    "VALID_PR_AUC": average_precision_score(y_valid, valid_proba_xgb),
    "VALID_ROC_AUC": roc_auc_score(y_valid, valid_proba_xgb),
    "TEST_PR_AUC": average_precision_score(y_test, test_proba_xgb),
    "TEST_ROC_AUC": roc_auc_score(y_test, test_proba_xgb),
    "Best_Threshold": best_thr_xgb,
    "TEST_Precision": test_metrics_xgb["precision"],
    "TEST_Recall": test_metrics_xgb["recall"],
    "TEST_F1": test_metrics_xgb["f1"]
})
pd.DataFrame(results)

threshold    0.980000
precision    0.498330
recall       0.427507
f1           0.460210
Name: 97, dtype: float64


,Model,VALID_PR_AUC,VALID_ROC_AUC,TEST_PR_AUC,TEST_ROC_AUC,Best_Threshold,TEST_Precision,TEST_Recall,TEST_F1
0,Cluster LogReg,0.341274,0.971044,0.471007,0.995102,0.990000,0.194444,0.856903,0.316965
1,Cluster XGB,0.464109,0.992541,0.415747,0.993946,0.980101,0.194444,0.856903,0.316965
2,Cluster XGB,0.464109,0.992541,0.415747,0.993946,0.980000,0.405930,0.443823,0.424032


In [21]:
pd.DataFrame(results)

,Model,VALID_PR_AUC,VALID_ROC_AUC,TEST_PR_AUC,TEST_ROC_AUC,Best_Threshold,TEST_Precision,TEST_Recall,TEST_F1
0,Cluster LogReg,0.341274,0.971044,0.471007,0.995102,0.99,0.194444,0.856903,0.316965
1,Cluster XGB,0.464109,0.992541,0.415747,0.993946,0.98,0.405930,0.443823,0.424032


По результатам мы видим, что видим, что модель показала не высокие результаты - примерно на уровне логистической регрессии с подобранными гиперпараметрами. Однак, важно тут не это. Мы доказали, что превращение всех признаков в метки кластеров и метки удаления от центроидов сохраняет полезный сигнал. Т.е. кластеризация успешно создала классы объектов, которые при их анализе позволяют выявить закономерности и предсказать мошенничество. 
Т.е., несмотря на то, что наша модель не показала высокой эффективности её можно было бы применить для инжиниринга признаков и обучения других моделях на данных + метках кластеров, что в целом могло бы стать интересным направлением развития борьбы с фродом.